# Accretion-ejection instability

$(\omega - m\Omega_\theta)^2 = \Omega_r^2 + \frac{2B_0 |k|}{\Sigma r} + \frac{k^2 c_s^2}{r^2}$

- $B_0$ campo imperturbato, prpendicolare al disco. legato al materiale del disco - assumiamo legge di potenza con esponente 5/4 come da Shakura&Sunyaev
- $\Sigma$ densità superficiale, legge di potenza con esponente ..
- assumo disco sottile, con aspect ratio $H/r = HOR$
- $c_s$ velocità del suono - in disco sottile è dato approssimativamente dalla formula sotto
- k = numero d'onda. lo prendiamo come parametro liberto ma poi controlliamo che stia in range ragionevole ($k \approx 1/H$ con H scala verticale tipica del disco)
- m: generalmente il modo eccitato ha un solo braccio, perciò poniamo m=1 per ridurre il numro di parametri


> $B_0 = B_{00} (r/r_{H})^{-\alpha_B}$. $B_{00}$ valore di riferimento sull'orizzonte degli eventi ($r_H$) del BH

> $\Sigma = \Sigma_0 (r/r_{in})^{-\alpha_\Sigma}$. $\Sigma_0$ riferimento al borndo interno del disco

> $c_s \approx (H/r) v_\varphi = (H/r) r \Omega_\varphi$

> $0.1/r < |k| < 10/r \ ,\ $ 
> $\beta = \frac{ 8 \pi p}{B^2_0} \approx \frac{ 8 \pi \Sigma c_s^2}{B^2_0} \leq 1 \ ,\ $
> $\frac{d}{dr}\left( \frac{\Omega\Sigma}{B^2_0} \right) > 0$

dato che l'equazione è una quadratica in |K| risolviamo analiticamente per omega=target. troviamo quindi 

$k(B_{00}, r, \Sigma_0, a)$

con $m = 1, h/r = HOR, \alpha = 0.5, M_{BH}$ fissati

## setup globale

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import pandas as pd
from setup import *
from plts_funcs import *

from AEI_setups.aei_common import (
    find_rossby, compute_disk_profile,
    mm, HOR, ALPHA_VISC
)

from AEI_setups.plot_disk_profiles import (
    plot_standard_4panels, summarize_comparison2,
    plot_full_disk_profiles,
    plot_aei_validity_map,
    plot_summary_table,
    plot_profiles_comparison,
)

from AEI_setups.radial_grid_analysis import (
    radial_scan_grid,
    plot_radial_grid,
    plot_validity_heatmap,
    compute_slopes_grid,
    plot_slope_distributions,
    summary_table_grid,
    compare_models,
)

from AEI_setups.diagnosis import (
    print_disk_boundaries,
    scan_disk_grid,
    plot_boundary_ratios,
    scan_boundary_grid,
    plot_boundary_distributions,
)

In [ ]:
param_grid = {
    'a':      np.linspace(-0.9, 0.9, 19),
    'B00':    np.logspace(-3, 8, 24),
    'Sigma0': np.logspace(0, 7, 16),
}
r_vec = np.geomspace(1, 1000, 100)

## v1 - profili omogenei su tutto il disco

### setup

In [ ]:
from AEI_setups.simple_disc import disk_model_simple

### zona B S&S (articolo 1)

In [ ]:
# ── v1: simple disc — find_rossby ────────────────────────────────────────
alp_B, alp_S = 5/4, 3/5   # esponenti standard

In [ ]:
base_1 = find_rossby(
    r_vec, param_grid,
    disk_model=lambda r, **p: disk_model_simple(r, **p, hr=HOR, M=M_BH,
                                                alpha_B=alp_B, alpha_S=alp_S),
    m=mm, hr=HOR, M=M_BH,
    check_k=False, check_beta=False, check_shear=False,
)

check_beta_1 = find_rossby(
    r_vec, param_grid,
    disk_model=lambda r, **p: disk_model_simple(r, **p, hr=HOR, M=M_BH,
                                                alpha_B=alp_B, alpha_S=alp_S),
    m=mm, hr=HOR, M=M_BH,
    check_k=False, check_beta=True, check_shear=False,
)

check_k_1 = find_rossby(
    r_vec, param_grid,
    disk_model=lambda r, **p: disk_model_simple(r, **p, hr=HOR, M=M_BH,
                                                alpha_B=alp_B, alpha_S=alp_S),
    m=mm, hr=HOR, M=M_BH,
    check_k=True, check_beta=False, check_shear=False,
)

In [ ]:
all_1 = find_rossby(
    r_vec, param_grid,
    disk_model=lambda r, **p: disk_model_simple(r, **p, hr=HOR, M=M_BH,
                                                alpha_B=alp_B, alpha_S=alp_S),
    m=mm, hr=HOR, M=M_BH,
    check_k=True, check_beta=True, check_shear=True,
)

print("=" * 80)
print("Tutti i check")
print("=" * 80)
plot_standard_4panels(all_1, "All checks - simple disc")

In [ ]:
dfs = {
    "Baseline": base_1,
    "Bound on k": check_k_1,
    "Bound on β": check_beta_1,
    "All bounds": all_1
}

comparison = summarize_comparison2(dfs)

#### analisi radiale

In [ ]:
df_all_s, df_binned_s, meta_list_s = radial_scan_grid(
    param_dict = param_grid,
    disk_model = lambda r, **p: disk_model_simple(r, **p, hr=HOR, M=M_BH,
                                                  alpha_B=alp_B, alpha_S=alp_S),
    mm=mm, hr=HOR,
    n_r     = 150,
    n_rbins = 30,
    r_max   = 1000.0,     # esplicito: simple_disc non ha r_BC
    verbose = True,
)

In [ ]:
fig = plot_radial_grid(df_binned_s, title="Simple disc — griglia parametri")
plt.show()

In [ ]:
# heatmap B00 vs Sigma0 su tutto il range radiale
fig = plot_validity_heatmap(df_all_s, param_x='B00', param_y='Sigma0')
plt.show()

# heatmap spin vs B00
fig = plot_validity_heatmap(df_all_s, param_x='a', param_y='B00')
plt.show()

In [ ]:
df_slopes_s = compute_slopes_grid(df_all_s)
fig = plot_slope_distributions(df_slopes_s)
plt.show()

In [ ]:
summary_table_grid(df_all_s, df_binned_s, df_slopes_s)

#### diagnostica

In [ ]:
# ── scegli il modello attivo ───────────────────────────────────────────────
active_model = lambda r, **p: disk_model_simple(r, **p,
                                             alpha_visc=ALPHA_VISC, hr=HOR, M=M_BH)

active_label = "simple"   # usato nei titoli dei grafici

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLA G1  ≈  cella D1:  raccoglie r_AB e r_BC su griglia di parametri
# ══════════════════════════════════════════════════════════════════════════

df_bounds = scan_boundary_grid(
    disk_model    = active_model,
    param_vectors = {
        'a':      np.linspace(-0.9, 0.9, 7),
        'Sigma0': np.logspace(0, 7, 8),
    },
    extra_params  = {'B00': 1e6},   # B00 non influisce su r_AB / r_BC
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLA G4:  distribuzione punti per zona in df_all
# ══════════════════════════════════════════════════════════════════════════
# Richiede che `df_all` sia già stato calcolato con find_rossby o
# radial_scan_grid. Sostituire `all_3` con il nome del tuo DataFrame.

df_check = all_1   # ← adattare se il nome è diverso

print(f"\n=== Distribuzione punti per zona in df_check ({active_label}) ===")
for zone in ['A', 'B', 'C']:
    sub = df_check[df_check['zone'] == zone]
    if sub.empty:
        print(f"  Zona {zone}: 0 punti")
        continue
    r_cols = 'r_norm' if 'r_norm' in sub.columns else 'r'
    print(f"  Zona {zone}: {len(sub):6d} punti  |  "
          f"r ∈ [{sub['r'].min():.1f}, {sub['r'].max():.0f}] rg")

### campo magnetico con andamento da osservazioni (articolo 2)

In [ ]:
alp_B = 1.7
alp_S = 3/5

In [ ]:
base_2 = find_rossby(
    r_vec, param_grid,
    disk_model=lambda r, **p: disk_model_simple(r, **p, hr=HOR, M=M_BH, 
                                                alpha_B=alp_B, alpha_S=alp_S),
    m=mm, hr=HOR, M=M_BH,
    check_k=False, check_beta=False, check_shear=False,
)

check_k_2 = find_rossby(
    r_vec, param_grid,
    disk_model=lambda r, **p: disk_model_simple(r, **p, hr=HOR, M=M_BH, 
                                                alpha_B=alp_B, alpha_S=alp_S),
    m=mm, hr=HOR, M=M_BH,
    check_k=True, check_beta=False, check_shear=False,
)

check_beta_2 = find_rossby(
    r_vec, param_grid,
    disk_model=lambda r, **p: disk_model_simple(r, **p, hr=HOR, M=M_BH, 
                                                alpha_B=alp_B, alpha_S=alp_S),
    m=mm, hr=HOR, M=M_BH,
    check_k=False, check_beta=True, check_shear=False,
)

all_2 = find_rossby(
    r_vec, param_grid,
    disk_model=lambda r, **p: disk_model_simple(r, **p, hr=HOR, M=M_BH, 
                                                alpha_B=alp_B, alpha_S=alp_S),
    m=mm, hr=HOR, M=M_BH,
    check_k=True, check_beta=True, check_shear=True,
)
print("=" * 80)
print("Tutti i check")
print("=" * 80)
plot_standard_4panels(all_2, "All checks - ")

In [ ]:
dfs = {
    "Baseline": base_2,
    "Bound on k": check_k_2,
    "Bound on β": check_beta_2,
    "All bounds": all_2
}

comparison = summarize_comparison2(dfs)

### profilo singolo

In [ ]:
# ── v1: profilo singolo ───────────────────────────────────────────────────
df_s1, meta_s1 = compute_disk_profile(
    disk_model = lambda r, **p: disk_model_simple(r, **p, hr=HOR, M=M_BH,
                                                  alpha_B=alp_B, alpha_S=alp_S),
    params     = {'a': 0.5, 'B00': 1e6, 'Sigma0': 1e5},
    mm=mm, hr=HOR,
    r_max=500.0,          # r_max esplicito: simple_disc non restituisce info['r_BC']
)
fig = plot_full_disk_profiles(df_s1, meta_s1, ALPHA_VISC)
plt.show()

## v2 - disco shakura sunyaev completo
implementazione delle tre regioni con raccordo di continuità tra le leggi di potenza, senza tutta la dipendenza precisa dai parametri termodfinamici del disco. 

si riconduce $\dot{M}$ a $\Sigma_0$ - ($\alpha, \Sigma_0, B_{00}$ ancora liberi)

### setup

In [ ]:
from AEI_setups.full_disk_SS import (
    disk_model_SS
)

### test singolo valore di parametri

In [ ]:
# ── v2: profilo singolo SS ────────────────────────────────────────────────
df_SS, meta_SS = compute_disk_profile(
    disk_model = lambda r, **p: disk_model_SS(r, **p, alpha_visc=ALPHA_VISC,
                                              hr=HOR, M=M_BH),
    params     = {'a': 0.5, 'B00': 1e6, 'Sigma0': 1e5},
    mm=mm, hr=HOR,
    # r_max auto: disk_model_SS restituisce info['r_BC']
)
print(f"r_AB = {meta_SS['r_AB']:.1f} rg   r_BC = {meta_SS['r_BC']:.0f} rg")
print(f"Punti AEI validi: {df_SS['aei_valid'].sum()} / {len(df_SS)}")

plot_summary_table(df_SS, meta_SS, ALPHA_VISC)

In [ ]:
fig = plot_full_disk_profiles(df_SS, meta_SS, ALPHA_VISC)
plt.show()

In [ ]:
fig2 = plot_aei_validity_map(df_SS, meta_SS)
plt.show()

### Test completo

#### Baseline: Nessun Check Fisico

In [ ]:
print("=" * 80)
print("BASELINE: Nessun check fisico (solo validità matematica e ISCO)")
print("=" * 80)

base_SS = find_rossby(
    r_vec, param_grid,
    disk_model=lambda r, **p: disk_model_SS(r, **p, alpha_visc=ALPHA_VISC, M=M_BH, hr=HOR),
    m=mm, hr=HOR, M=M_BH,
    check_k=False, check_beta=False, check_shear=False,
)
plot_standard_4panels(base_SS, "Baseline - ")

#### Check 1: Solo Bound su k

In [ ]:
print("\n" + "=" * 80)
print("CHECK 1: Solo bound fisico su k (0.1/r < k < 10/r)")
print("=" * 80)

check_k_SS = find_rossby(
    r_vec, param_grid,
    disk_model=lambda r, **p: disk_model_SS(r, **p, alpha_visc=ALPHA_VISC, M=M_BH, hr=HOR),
    m=mm, hr=HOR, M=M_BH,
    check_k=True, check_beta=False, check_shear=False,
)

plot_standard_4panels(check_k_SS, "Check k - ")

print(f"\nRiduzione rispetto a baseline: {len(base_SS)} → {len(check_k_SS)} "
      f"({len(check_k_SS)/len(base_SS)*100:.1f}%)")

#### Check 2: Solo Bound su β

In [ ]:
print("\n" + "=" * 80)
print("CHECK 2: Solo bound su β (β ≤ 1 per instabilità AEI)")
print("=" * 80)

check_beta_SS = find_rossby(
    r_vec, param_grid,
    disk_model=lambda r, **p: disk_model_SS(r, **p, alpha_visc=ALPHA_VISC, M=M_BH, hr=HOR),
    m=mm, hr=HOR, M=M_BH,
    check_k=False, check_beta=True, check_shear=False,
)

plot_standard_4panels(check_beta_SS, "Check β - ")

print(f"\nRiduzione rispetto a baseline: {len(base_SS)} → {len(check_beta_SS)} "
      f"({len(check_beta_SS)/len(base_SS)*100:.1f}%)")

#### Check 3: Solo Bound su Shear

In [ ]:
print("\n" + "=" * 80)
print("CHECK 3: Solo bound su shear (d/dr[Ωφ × Σ / B₀²] > 0)")
print("=" * 80)

check_shear_SS = find_rossby(
    r_vec, param_grid,
    disk_model=lambda r, **p: disk_model_SS(r, **p, alpha_visc=ALPHA_VISC, M=M_BH, hr=HOR),
    m=mm, hr=HOR, M=M_BH,
    check_k=False, check_beta=False, check_shear=True,
)

plot_standard_4panels(check_shear_SS, "Check Shear - ")

print(f"\nRiduzione rispetto a baseline: {len(base_SS)} → {len(check_shear_SS)} "
      f"({len(check_shear_SS)/len(base_SS)*100:.1f}%)")

#### Tutti i Check Combinati

In [ ]:
print("\n" + "=" * 80)
print("TUTTI I CHECK COMBINATI")
print("=" * 80)

all_SS = find_rossby(
    r_vec, param_grid,
    disk_model=lambda r, **p: disk_model_SS(r, **p, alpha_visc=ALPHA_VISC, M=M_BH, hr=HOR),
    m=mm, hr=HOR, M=M_BH,
    check_k=True, check_beta=True, check_shear=True,
)

plot_standard_4panels(all_SS, "All Checks - ")

print(f"\nRiduzione rispetto a baseline: {len(base_SS)} → {len(all_SS)} "
      f"({len(all_SS)/len(base_SS)*100:.1f}%)")

#### Confronto Riassuntivo

In [ ]:
comparison_SS = summarize_comparison2({
    'Baseline': base_SS,
    'Bound on k': check_k_SS,
    'Bound on β': check_beta_SS,
    'Bound on shear': check_shear_SS,
    'All checks': all_SS,
})

### Analisi radiale

In [ ]:
param_dict_SS = {
    'a':      (-0.9, 0.9, 7),
    'B00':    (1e1,  1e8, 8),
    'Sigma0': (1e3,  1e7, 8),
}

df_all_SS, df_binned_SS, meta_list_SS = radial_scan_grid(
    param_dict = param_dict_SS,
    disk_model = lambda r, **p: disk_model_SS(r, **p, alpha_visc=ALPHA_VISC,
                                              hr=HOR, M=M_BH),
    mm=mm, hr=HOR,
    n_r     = 150,
    n_rbins = 30,
    # r_max auto: disk_model_SS restituisce info['r_BC']
    verbose = True,
)

In [ ]:
fig = plot_radial_grid(df_binned_SS, title="Full disk S&S — griglia parametri")
plt.show()

In [ ]:
# heatmap per zona
for zone in ['A', 'B', 'C']:
    fig = plot_validity_heatmap(
        df_all_SS, param_x='B00', param_y='Sigma0', zone=zone
    )
    plt.show()

In [ ]:
# heatmap con range radiale ristretto — solo zona interna (r < 50 rg)
fig = plot_validity_heatmap(
    df_all_SS, param_x='a', param_y='Sigma0',
    r_range=(None, 50), metric='aei_valid'
)
plt.show()

In [ ]:
df_slopes_SS = compute_slopes_grid(df_all_SS)
fig = plot_slope_distributions(df_slopes_SS)
plt.show()

summary_table_grid(df_all_SS, df_binned_SS, df_slopes_SS)

### diagnostica

In [ ]:
# ── scegli il modello attivo ───────────────────────────────────────────────
active_model = lambda r, **p: disk_model_SS(r, **p,
                                             alpha_visc=ALPHA_VISC, hr=HOR, M=M_BH)

active_label = "SS"   # usato nei titoli dei grafici

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLA G1  ≈  cella D1:  raccoglie r_AB e r_BC su griglia di parametri
# ══════════════════════════════════════════════════════════════════════════

df_bounds = scan_boundary_grid(
    disk_model    = active_model,
    param_vectors = {
        'a':      np.linspace(-0.9, 0.9, 7),
        'Sigma0': np.logspace(0, 7, 8),
    },
    extra_params  = {'B00': 1e6},   # B00 non influisce su r_AB / r_BC
)

In [ ]:
fig = plot_boundary_distributions(df_bounds, param_color='a')
fig.suptitle(f"Distribuzione frontiere — modello {active_label}", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLA G3:  run con r_AB troppo vicino a r_ISCO (zona A nana)
# ══════════════════════════════════════════════════════════════════════════

soglia = 1.6   # r_AB < soglia * r_ISCO → zona A è una striscia minima
degeneri = df_bounds[df_bounds['r_AB_over_rISCO'] < soglia]
print(f"\nRun con r_AB < {soglia} × r_ISCO: {len(degeneri)} / {len(df_bounds)}"
      f"  ({len(degeneri)/len(df_bounds)*100:.0f}%)")
print(degeneri[['a', 'Sigma0', 'r_AB', 'r_ISCO']].head(10))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLA G4:  distribuzione punti per zona in df_all
# ══════════════════════════════════════════════════════════════════════════
# Richiede che `df_all` sia già stato calcolato con find_rossby o
# radial_scan_grid. Sostituire `all_3` con il nome del tuo DataFrame.

df_check = all_SS   # ← adattare se il nome è diverso

print(f"\n=== Distribuzione punti per zona in df_check ({active_label}) ===")
for zone in ['A', 'B', 'C']:
    sub = df_check[df_check['zone'] == zone]
    if sub.empty:
        print(f"  Zona {zone}: 0 punti")
        continue
    r_cols = 'r_norm' if 'r_norm' in sub.columns else 'r'
    print(f"  Zona {zone}: {len(sub):6d} punti  |  "
          f"r ∈ [{sub['r'].min():.1f}, {sub['r'].max():.0f}] rg")

In [ ]:
params_test = {'a': 0.2, 'B00': 1e7, 'Sigma0': 1e3}

df_test, meta_test = compute_disk_profile(
    disk_model = active_model,
    params     = params_test,
    mm=mm, hr=HOR,
)

print(f"\nRun di test  ({active_label}):  a={params_test['a']},  "
      f"B00={params_test['B00']:.0e},  Σ0={params_test['Sigma0']:.0e}")
print(f"  r_ISCO = {meta_test['r_ISCO']:.2f} rg")
if 'r_AB' in meta_test:
    print(f"  r_AB   = {meta_test['r_AB']:.2f} rg")
    print(f"  r_BC   = {meta_test['r_BC']:.0f} rg")
print(f"\n  Punti per zona:")
for zone in ['A', 'B', 'C']:
    sub = df_test[df_test['zone'] == zone]
    if len(sub):
        print(f"    Zona {zone}: {len(sub)} punti, "
              f"r ∈ [{sub['r'].min():.2f}, {sub['r'].max():.1f}] rg")

fig = plot_full_disk_profiles(df_test, meta_test, ALPHA_VISC)
plt.suptitle(f"Profilo disco — modello {active_label}", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLA G6  ≈  cella 52:  diagnostica grafica delle funzioni di crossing
# ══════════════════════════════════════════════════════════════════════════
# Plotta β/(1-β) di default. Per aggiungere condizioni custom, passare
# `condition_funcs` come dict {label: callable(r, B0, Sigma, c_s) -> array}.

fig = plot_boundary_ratios(
    disk_model = active_model,
    params     = params_test,
    r_range    = None,    # None → [r_ISCO*1.01, 1e5]
    n_points   = 500,
)
fig.suptitle(f"Crossing conditions — modello {active_label}", y=1.02)
plt.show()


In [ ]:
print_disk_boundaries(active_model, params_test, M=M_BH)

In [ ]:
df_grid = scan_disk_grid(
    disk_model   = active_model,
    Sigma0_vals  = [1e3, 1e4, 1e5, 1e6, 1e7],
    a_vals       = [-0.9, 0.0, 0.5, 0.9],
    extra_params = {'B00': 1e6},
)

## v3 - disco S&S con correzioni NT complete

calcolo esatto degli andamenti in ogni zona per B e Sigma

### setup

In [ ]:
from AEI_setups.ss_nt_boundaries import (
    disk_model_NT,
    nt_print_boundaries,
    nt_scan_grid,
)

### test griglia

In [ ]:
base_NT = find_rossby(
    r_vec      = r_vec,
    param_grid = param_grid,
    disk_model = lambda r, **p: disk_model_NT(
                     r, **p, alpha_visc=ALPHA_VISC, hr=HOR, M=M_BH),
    m=mm, hr=HOR, M=M_BH,
    check_k=False, check_beta=False, check_shear=False,
)
plot_standard_4panels(base_NT, "Baseline NT - ")

In [ ]:
all_NT = find_rossby(
    r_vec      = r_vec,
    param_grid = param_grid,
    disk_model = lambda r, **p: disk_model_NT(
                     r, **p, alpha_visc=ALPHA_VISC, hr=HOR, M=M_BH),
    m=mm, hr=HOR, M=M_BH,
    check_k=True, check_beta=True, check_shear=True,
)
plot_standard_4panels(all_NT, "All checks NT - ")

In [ ]:
comparison_NT = summarize_comparison2({
    'Baseline': base_NT,
    'All checks': all_NT,
})

### test sdingolo

In [ ]:
# ── v3: NT — profilo singolo ──────────────────────────────────────────────
df_NT, meta_NT = compute_disk_profile(
    disk_model = lambda r, **p: disk_model_NT(r, **p, alpha_visc=ALPHA_VISC,
                                              hr=HOR, M=M_BH),
    params     = {'a': 0.5, 'B00': 1e7, 'Sigma0': 1e2},
    mm=mm, hr=HOR,
    # r_max auto: disk_model_NT restituisce info['r_BC']
)
nt_print_boundaries(a=0.5, Sigma0=1e3, alpha=ALPHA_VISC)
print(f"Punti AEI validi: {df_NT['aei_valid'].sum()} / {len(df_NT)}")

fig = plot_full_disk_profiles(df_NT, meta_NT, ALPHA_VISC)
plt.show()

### analisi radiale

In [ ]:
df_all_NT, df_binned_NT, meta_list_NT = radial_scan_grid(
    param_dict = param_grid,
    disk_model = lambda r, **p: disk_model_NT(r, **p, alpha_visc=ALPHA_VISC,
                                              hr=HOR, M=M_BH),
    mm=mm, hr=HOR,
    n_r     = 150,
    n_rbins = 30,
    verbose = True,
)

In [ ]:
fig = plot_radial_grid(df_binned_NT, title="NT corrections — griglia parametri")
plt.show()

In [ ]:
df_slopes_NT = compute_slopes_grid(df_all_NT)
fig = plot_slope_distributions(df_slopes_NT)
plt.show()

summary_table_grid(df_all_NT, df_binned_NT, df_slopes_NT)

### diagnostica

In [ ]:
# ── scegli il modello attivo ───────────────────────────────────────────────
active_model = lambda r, **p: disk_model_NT(r, **p,
                                             alpha_visc=ALPHA_VISC, hr=HOR, M=M_BH)

active_label = "NT"   # usato nei titoli dei grafici

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLA G1  ≈  cella D1:  raccoglie r_AB e r_BC su griglia di parametri
# ══════════════════════════════════════════════════════════════════════════

df_bounds = scan_boundary_grid(
    disk_model    = active_model,
    param_vectors = {
        'a':      np.linspace(-0.9, 0.9, 7),
        'Sigma0': np.logspace(0, 7, 8),
    },
    extra_params  = {'B00': 1e7},   # B00 non influisce su r_AB / r_BC
)

In [ ]:
fig = plot_boundary_distributions(df_bounds, param_color='a')
fig.suptitle(f"Distribuzione frontiere — modello {active_label}", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLA G3:  run con r_AB troppo vicino a r_ISCO (zona A nana)
# ══════════════════════════════════════════════════════════════════════════

soglia = 1.6   # r_AB < soglia * r_ISCO → zona A è una striscia minima
degeneri = df_bounds[df_bounds['r_AB_over_rISCO'] < soglia]
print(f"\nRun con r_AB < {soglia} × r_ISCO: {len(degeneri)} / {len(df_bounds)}"
      f"  ({len(degeneri)/len(df_bounds)*100:.0f}%)")
print(degeneri[['a', 'Sigma0', 'r_AB', 'r_ISCO']].head(10))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLA G4:  distribuzione punti per zona in df_all
# ══════════════════════════════════════════════════════════════════════════
# Richiede che `df_all` sia già stato calcolato con find_rossby o
# radial_scan_grid. Sostituire `all_3` con il nome del tuo DataFrame.

df_check = all_NT   # ← adattare se il nome è diverso

print(f"\n=== Distribuzione punti per zona in df_check ({active_label}) ===")
for zone in ['A', 'B', 'C']:
    sub = df_check[df_check['zone'] == zone]
    if sub.empty:
        print(f"  Zona {zone}: 0 punti")
        continue
    r_cols = 'r_norm' if 'r_norm' in sub.columns else 'r'
    print(f"  Zona {zone}: {len(sub):6d} punti  |  "
          f"r ∈ [{sub['r'].min():.1f}, {sub['r'].max():.0f}] rg")

In [ ]:
params_test = {'a': 0.2, 'B00': 1e7, 'Sigma0': 1e3}

df_test, meta_test = compute_disk_profile(
    disk_model = active_model,
    params     = params_test,
    mm=mm, hr=HOR,
)

print(f"\nRun di test  ({active_label}):  a={params_test['a']},  "
      f"B00={params_test['B00']:.0e},  Σ0={params_test['Sigma0']:.0e}")
print(f"  r_ISCO = {meta_test['r_ISCO']:.2f} rg")
if 'r_AB' in meta_test:
    print(f"  r_AB   = {meta_test['r_AB']:.2f} rg")
    print(f"  r_BC   = {meta_test['r_BC']:.0f} rg")
print(f"\n  Punti per zona:")
for zone in ['A', 'B', 'C']:
    sub = df_test[df_test['zone'] == zone]
    if len(sub):
        print(f"    Zona {zone}: {len(sub)} punti, "
              f"r ∈ [{sub['r'].min():.2f}, {sub['r'].max():.1f}] rg")

fig = plot_full_disk_profiles(df_test, meta_test, ALPHA_VISC)
plt.suptitle(f"Profilo disco — modello {active_label}", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLA G6  ≈  cella 52:  diagnostica grafica delle funzioni di crossing
# ══════════════════════════════════════════════════════════════════════════
# Plotta β/(1-β) di default. Per aggiungere condizioni custom, passare
# `condition_funcs` come dict {label: callable(r, B0, Sigma, c_s) -> array}.

fig = plot_boundary_ratios(
    disk_model = active_model,
    params     = params_test,
    r_range    = None,    # None → [r_ISCO*1.01, 1e5]
    n_points   = 500,
)
fig.suptitle(f"Crossing conditions — modello {active_label}", y=1.02)
plt.show()


In [ ]:
print_disk_boundaries(active_model, params_test, M=M_BH)

In [ ]:
df_grid = scan_disk_grid(
    disk_model   = active_model,
    Sigma0_vals  = [1e3, 1e4, 1e5, 1e6, 1e7],
    a_vals       = [-0.9, 0.0, 0.5, 0.9],
    extra_params = {'B00': 1e6},
)

## Confronto radiale

In [ ]:
# compare_models sovrappone le mediane radiali di tutti i modelli
fig = compare_models({
    'Simple': (df_all_s,  df_binned_s,  meta_list_s),
    'SS':     (df_all_SS, df_binned_SS, meta_list_SS),
    'NT':     (df_all_NT, df_binned_NT, meta_list_NT),
})
plt.show()

In [ ]:
# confronto tabellare — frac_AEI per zona, per modello
for name, (df_all, df_binned, _) in {
    'Simple': (df_all_s,  df_binned_s,  None),
    'SS':     (df_all_SS, df_binned_SS, None),
    'NT':     (df_all_NT, df_binned_NT, None),
}.items():
    print(f"\n{'='*20}  {name}  {'='*20}")
    summary_table_grid(df_all, df_binned)